<a href="https://colab.research.google.com/github/ikramkakar/Build-a-Retrieval-Augmented-Generation-RAG-/blob/main/Retrieval_Augmented_Generation_(RAG)_Project_using_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We will build a RAG pipeline using LangChain to enable intelligent question-answering over legal documents and agreements. The project includes document ingestion, chunking, embeddings, vector database creation, retrieval, and answer generation using LLMs. In order to complete this project I would need to install following required libararies :

***!pip install -q langchain***

*   LangChain Framework
*   Used to build the complete RAG pipeline (document loading, chunking, retrieval, QA)

***!pip install -q langchain-community***

*  LangChain Community Integrations
*  Provides integrations for:PDF loaders, FAISS, HuggingFace embeddings, etc.

***!pip install -q langchain-openai***


*   OpenAI Integration for LangChain
*   Used for OpenAI embeddings and GPT models

***!pip install -q langchain-text-splitters***
*   Text Splitters
*   Used to split large legal documents into smaller chunks for better retrieval and embedding performance

!pip install -q pypdf
*    PDF Reader Library
*   Used by PyPDFLoader to read PDF legal documents


***!pip install -q chromadb***


*   ChromaDB Vector Database
*   Stores embeddings for semantic retrieval

***!pip install -q openai***
*   OpenAI SDK
*  Connects to OpenAI API

***!pip install -q tiktoken***
*   Tokenizer Library
*  Helps count tokens and process text

In [9]:

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-openai
!pip install -q langchain-text-splitters
!pip install -q pypdf
!pip install -q chromadb
!pip install -q openai
!pip install -q tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

import os


*  It isa command in Python that loads the built-in operating system module, allowing your code to interact with the underlying operating system (Windows, macOS, or Linux). It is used for file manipulation
from google.colab import files



*   Upload files in Colab

from langchain_community.document_loaders import PyPDFLoader
*  PDF Loader

from langchain_text_splitters import RecursiveCharacterTextSplitter
*   Text Splitter

from langchain_openai import OpenAIEmbeddings

*   OpenAI Embeddings

from langchain_community.vectorstores import Chroma
*   Chroma Vector Database

from pprint import ppri
* Pretty Print

In [10]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from pprint import pprint

Set my OpenAI key

In [27]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

Load my PDF file-
There are two way to load the files


*   I created a folder and uploaded all the documents into the folder and then load then using following code- I right click on it and copy the page so I can have the whole files in the directory
*   Another Technique is , we load one document at time, Chunk it, Embed it and then store it in vector Database and then retrieve it .

I will upload all the document in one directory



In [28]:
pdf_directory = "/content/LegalContractsAgreements"

documents = []
for file_name in os.listdir(pdf_directory):

    if file_name.endswith(".pdf"):
        pdf_path = os.path.join(pdf_directory, file_name)
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        documents.extend(docs)

        print(f"Loaded: {file_name}")

print("\nTotal Pages Loaded:", len(documents))

Loaded: Doc1.pdf
Loaded: Doc3.pdf
Loaded: Doc2.pdf
Loaded: Doc4.pdf
Loaded: Doc5.pdf

Total Pages Loaded: 16


To split Documents into Chunks I will use RecursiveCharacterTextSplitter

In [29]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 35


tO Create Embeddings and Store in VectorDB (ChromaDB)

In [30]:
if chunks:
    embeddings = OpenAIEmbeddings()

    vectorstore = Chroma.from_documents(chunks, embeddings)
    print("Embeddings created and stored in ChromaDB.")
else:
    print("No chunks available to create embeddings. Please check chunking step.")
    vectorstore = None

Embeddings created and stored in ChromaDB.


Set Up and Query the RAG Chain-Build RAG Question-Answering Pipeline

In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

In [34]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = vectorstore.as_retriever()
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context:\n\n{context}\n\nQuestion: {question}"
)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
query = "What is this document about?"
response = rag_chain.invoke(query)

print(response)

The document appears to be a legal contract or agreement, as indicated by its source title "LegalContractsAgreements" and the presence of spaces for dates, which are typically included in such documents for signatures or formal acknowledgments. However, the specific content or details of the agreement are not provided in the context.


Semantic search

In [33]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever Ready")

Retriever Ready


Questions: Employment Agreement — Notice Period

In [55]:
new_query =  new_query = """
"What is the notice period in the employment agreement?"
Instructions:
- Answer in bullet points """

if 'rag_chain' in locals() and rag_chain:
    new_response = rag_chain.invoke(new_query)
    print("\nAnswer from updated RAG chain:")
    print(new_response)
else:
    print("RAG chain not initialized. Please ensure previous steps ran successfully.")


Answer from updated RAG chain:
- The employment agreement is AT-WILL, allowing either party to terminate employment at any time with or without cause.
- Notice provisions include:
  - Termination by Employer without cause: 30 days written notice OR payment of 30 days base salary in lieu of notice.
  - Resignation by Employee: 30 days written notice to the Employer.
  - Termination for cause (gross misconduct, fraud, breach of confidentiality): Immediate termination without notice or pay in lieu.


Notice Period

In [39]:
new_query = """
What is the notice period in the employment agreement?

Instructions:
- Answer in bullet points
"""


if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Query: 
What is the notice period in the employment agreement?

Instructions:
- Answer in bullet points


Answer:
- The employment agreement is AT-WILL, allowing either party to terminate employment at any time with or without cause.
- Notice provisions include:
  - Termination by Employer without cause: 30 days written notice OR payment of 30 days base salary in lieu of notice.
  - Resignation by Employee: 30 days written notice to the Employer.
  - Termination for cause (gross misconduct, fraud, breach of confidentiality): Immediate termination without notice or pay in lieu.


Non-Compete Claus

In [40]:
new_query = """
Is there a non-compete clause in the employment agreement?

Instructions:
- Answer in bullet points
"""


if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Query: 
Is there a non-compete clause in the employment agreement?

Instructions:
- Answer in bullet points


Answer:
- Yes, there is a non-compete clause in the employment agreement.
- The non-compete clause is effective during the term of employment and for a period of twelve (12) months following termination for any reason.
- It prohibits the employee from:
  - Working for, consulting with, or holding equity in any direct competitor in the predictive analytics or business intelligence software sector within the United States.
  - Soliciting or attempting to solicit any of the employer's clients, prospective clients, or business partners that the employee had material contact with during employment.
  - Engaging in any activity that would constitute a conflict of interest with the employer's business operations.
- The geographic scope of the non-compete is the continental United States.


NDA — Confidential Information

In [41]:
new_query = """
What information is considered confidential in the NDA?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Query: 
What information is considered confidential in the NDA?

Instructions:
- Answer in bullet points


Answer:
- Business plans
- Marketing strategies
- Financial information
- Pricing models
- Operational processes


NDA — Confidentiality Exceptions

In [42]:
new_query = """
What are the confidentiality exceptions in the NDA?

Instructions:
- Answer in bullet points
"""


if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Query: 
What are the confidentiality exceptions in the NDA?

Instructions:
- Answer in bullet points


Answer:
The confidentiality exceptions in the NDA are:

- Information that is or becomes publicly available through no act or omission of the Receiving Party.
- Information that was already known to the Receiving Party at the time of disclosure, as evidenced by written records predating the disclosure.
- Information that is rightfully received from a third party who is not under any confidentiality obligation and who did not obtain it from the Disclosing Party.
- Information that is independently developed by the Receiving Party without reference to or use of the Confidential Information.


SLA — Guaranteed Uptime

In [43]:
new_query = """
What is the guaranteed uptime in the SLA?

Instructions:
- Answer in bullet points
"""


if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Query: 
What is the guaranteed uptime in the SLA?

Instructions:
- Answer in bullet points


Answer:
- Production (Tier 1): 99.95%
- Staging (Tier 2): 99.50%
- Development (Tier 3): 99.00%


SLA — Penalties for SLA Breach

In [54]:
new_query = """
What are the penalties for SLA breach?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Answer:
- Service credits are the Client's sole and exclusive remedy for SLA breaches.
- Credits are applied to the next invoice and are not redeemable for cash.
- Total credits in any month shall not exceed 100% of the monthly service fee.
- Response time SLA breaches (P1 unresolved beyond 4 hours) attract a flat penalty of USD 500 per hour of breach beyond the resolution target.


Vendor Agreement — Payment Terms

In [53]:
new_query = """
What are the payment terms in the vendor agreement?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Answer:
- Buyer may terminate for convenience with 60 days written notice.
- Upon termination for convenience, the Buyer must pay all amounts due for work completed.
- A termination fee of 15% of the remaining contract value is applicable.


Payment Terms

In [46]:
new_query = """
What are the payment terms in the vendor agreement?

Instructions:
- Answer in bullet points
"""

print(f"\nQuery: {new_query}")

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Query: 
What are the payment terms in the vendor agreement?

Instructions:
- Answer in bullet points


Answer:
- Buyer may terminate for convenience with 60 days written notice.
- Upon termination for convenience, the Buyer must pay all amounts due for work completed.
- A termination fee of 15% of the remaining contract value is also required upon termination for convenience.


Vendor Agreement — Warranty Support

In [52]:
new_query = """
Is warranty support included in the vendor agreement?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Answer:
- Yes, warranty support is included in the vendor agreement.
- The warranty support described in clauses (a) and (b) is included in the contract price.
- There are no additional charges for the warranty support.


Vendor Agreement — Warranty Support

In [49]:
new_query = """
Is warranty support included in the vendor agreement?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Answer:
- Yes, warranty support is included in the vendor agreement.
- The warranty support described in clauses (a) and (b) is included in the contract price.
- There are no additional charges for the warranty support.


Lease Agreement — Monthly Rent

In [50]:
new_query = """
What is the monthly rent in the lease agreement?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Answer:
- The document does not specify the monthly rent amount in the lease agreement.


Maintenance Costs

In [51]:
new_query = """
Who handles maintenance costs in the lease agreement?

Instructions:
- Answer in bullet points
"""

if 'rag_chain' in locals() and rag_chain:

    new_response = rag_chain.invoke(new_query)

    print("\nAnswer:")
    print(new_response)

else:
    print("RAG chain not initialized.")


Answer:
- **Landlord's Responsibilities:**
  - Structural repairs (walls, columns, beams, roof, and building facade)
  - Common area maintenance (lobbies, corridors, restrooms, parking, landscaping)
  - Maintenance of lifts, escalators, and building-wide HVAC and fire suppression systems
  - External plumbing and electrical infrastructure up to the Premises entry point
  - Pest control for common areas and building exterior

- **Tenant's Responsibilities:**
  - Maintenance of interior fixtures, furniture, fittings, and tenant-specific installations
  - Electrical wiring, switches, and equipment within the Premises
  - Interior painting and cosmetic upkeep of the Premises
  - Maintenance of air-conditioning units serving exclusively the Premises
  - Any damage caused by Tenant's employees, clients, or contractors
  - Pest control within the Premises
